# Friends from proximity

Can you tell who someone's friends are just from which phones are near theirs?

## Setup

In [2]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../src")

import pandas as pd
from mlxtend.frequent_patterns import association_rules, fpgrowth
import preprocess as pp
import friends as fr

## The data and baseline

The proximity data and student table are from the preprocessing notebook. We only use the students who appear in both Bluetooth and Facebook data, since a student without Facebook data is missing data, not necessarily a student without friends.

Two students picked randomly from this group are friends on Facebook with 2.11% probability. Every result obtained is compared against that baseline.

In [3]:
prox = pp.proximity()
st = pp.student_table()
fb = pp.clean_fb(pp.load_fb())

# students in a real pair and in the friendship file
users = set(st.index[st.in_bt & st.in_fb])
friends = fr.friend_pairs(fb, users)
n = len(users)
# pairs of students that could be friends
possible = n * (n - 1) / 2
baseline = len(friends) / possible

print(f"{len(prox):,} proximity rows, {n} students, {len(friends):,} friendships, baseline {baseline:.2%}")

2,426,275 proximity rows, 657 students, 4,551 friendships, baseline 2.11%


## What counts as together

Before mining, we need to decide how storng the signal must be and which hours to use.

**Signal strength.** RSSI has no gap that separates "at the same table" from "in the same building" (see RSSI plot in `preprocess.ipynb`). We decide that two students are together when the signal is **-80** dBm or stronger, that keeps the strongest ~30% of readings. But we will also test -75 and -85 as well to see if there are better picks.

**Teaching hours.** One weekdays the number of rows increase drastically at 08:00, and falls of at 17:00. We call weekdays between 8-17 teaching hours, and the overwhelming majority of readings are withing theses hours.

In [4]:
rssi = -80
min_count = 100

In [5]:
print("RSSI 70% and 75% quantiles:", prox.rssi.quantile([0.7, 0.75]).to_list())
print(f"rows at {rssi} or stronger: {(prox.rssi >= rssi).mean():.0%}")

weekday = prox[~(prox.timestamp // pp.DAY % 7).isin([0, 6])]
by_hour = ((weekday.timestamp % pp.DAY) // 3600).value_counts().sort_index()
print("weekday rows at hours 7, 8, 16, 17:", by_hour[[7, 8, 16, 17]].to_list())
print(f"rows in teaching hours: {fr.in_class(prox).mean():.0%}")

RSSI 70% and 75% quantiles: [-80.0, -79.0]
rows at -80 or stronger: 30%
weekday rows at hours 7, 8, 16, 17: [34625, 175029, 105225, 44947]
rows in teaching hours: 71%


In a lecture, the schedule puts students together, not their choice. Below we can see what happens when we exclude the teaching hours. From the pairs who are near at least once in class, 9.2% are facebook friends; but from the pairs who are near at least once outside of class, 15.1% are.

Therefore, time outside of class says more about friendship. Thus, we use those hours. But we also run on the teaching hours to compare.

In [6]:
near_share = {}
for teaching in [True, False]:
    p = fr.together(prox, rssi, teaching)
    near = {s for s in map(frozenset, zip(p.user_a, p.user_b)) if s.issubset(users)}
    near_share[teaching] = len(near.intersection(friends)) / len(near)
    print(f"teaching {teaching}: {len(p):,} rows, {len(near):,} pairs, {near_share[teaching]:.1%} friends")

teaching True: 503,022 rows, 24,600 pairs, 9.2% friends
teaching False: 228,469 rows, 10,927 pairs, 15.1% friends


## Transactions

Frequest itemset mining requires transactions, like items in ones shopping bag. For us, one transction is one student in one hour and the items are that student and everyone close to them in that hour. For example, if student 2 is near students 4 and 8 between 19:00-20:00, then the transaction is {2, 4, 8}.

The student is part of their own transaction, otherwise two students (2 and 4 from above) would give the the transactions {2} and {4}, so the pair would never appear in the same transaction as {2, 4}.

**1h window:** With a short time window, one group could be split into several smaller transcations. One hour has 12 scans (5-min scans) so a student only needs to be seen in one of them to be in the transaction. An hour is still short enough to keep two events of the same evening apart, e.g. dinner and party.

There's 50,874 transactions. Half only hold two students and 99% hold 16 or fewer, so the groups stay small when teaching hours are not included.

In [7]:
pairs = fr.together(prox, rssi)
X = fr.transactions(pairs)
sizes = X.sum(axis=1)

print(f"{len(X):,} transactions, {X.shape[1]} students")
print("students per transaction (50%, 75%, 99%):", sizes.quantile([0.5, 0.75, 0.99]).to_list(), "largest", sizes.max())

50,874 transactions, 664 students
students per transaction (50%, 75%, 99%): [2.0, 3.0, 16.0] largest 43
